# 02 — Market-consistent feature construction

This notebook transforms the cleaned hourly data into the common supervised-learning samples used by the machine-learning and deep-learning models in the paper.

For every eligible delivery day $D$, one sample is created at **11:00 local time on $D-1$**. The sample contains:

- the 168 most recent published day-ahead prices ending at 23:00 on $D-1$;
- four origin-available summaries for each of seven selected exogenous variables;
- five deterministic calendar descriptors for delivery day $D$; and
- the 24 hourly prices for delivery day $D$ as the target vector.

The notebook constructs features and targets only. It does **not** scale inputs, fit models, select hyperparameters or inspect 2018 performance.

### Input

- `data/processed/energy_dataset_clean.csv`

### Outputs

- `data/features/day_ahead_features.npz`
- `data/features/sample_metadata.csv`
- `data/features/target_timestamps.csv`
- `data/features/tabular_feature_names.csv`
- `data/features/sequence_feature_names.csv`
- `outputs/audits/feature_construction_summary.csv`
- `outputs/audits/excluded_delivery_days.csv`

## 1. Configuration and imports

In [3]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
from IPython.display import display

TIMEZONE = "Europe/Paris"  # CET/CEST convention used in the manuscript
TIME_COLUMN = "time"
TARGET = "price"
HORIZON = 24
PRICE_WINDOW = 168
CUTOFF_HOUR = 11

EXOGENOUS_VARIABLES = [
    "total_load_actual",
    "gen_wind_onshore",
    "gen_solar",
    "gen_fossil_gas",
    "gen_hydro_pumped_storage_consumption",
    "humidity",
    "temperature",
]

CALENDAR_FEATURES = [
    "delivery_dow_sin",
    "delivery_dow_cos",
    "delivery_month_sin",
    "delivery_month_cos",
    "delivery_is_weekend",
]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "energy_dataset_clean.csv"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
AUDIT_DIR = PROJECT_ROOT / "outputs" / "audits"
FEATURE_FILE = FEATURE_DIR / "day_ahead_features.npz"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Project root: {PROJECT_ROOT}")

Python: 3.12.2
pandas: 2.2.2
NumPy: 1.26.4
Project root: /Users/fatimarodrigues/Documents/Aulas/MINDD/Trabs-Pratico/2025-26/TP2-2025/git-price-forecasting/electricity-price-forecasting-revisão/initial-weather


## 2. Load and validate the cleaned hourly data

Timestamp continuity is checked in UTC so that the spring and autumn daylight-saving transitions are represented correctly.

In [15]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {INPUT_FILE}\n"
        "Run notebooks/01_data_preparation.ipynb first."
    )

df = pd.read_csv(INPUT_FILE)
if TIME_COLUMN not in df.columns:
    raise KeyError(f"The cleaned dataset does not contain '{TIME_COLUMN}'.")

df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], utc=True, errors="raise")
df[TIME_COLUMN] = df[TIME_COLUMN].dt.tz_convert(TIMEZONE)
df = df.set_index(TIME_COLUMN).sort_index()

required_columns = [TARGET, *EXOGENOUS_VARIABLES]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required variables: {missing_columns}")
if df.index.has_duplicates:
    raise ValueError("The cleaned dataset contains duplicate timestamps.")
if df[required_columns].isna().any().any():
    raise ValueError("Required modelling variables contain missing values.")

utc_steps = df.index.tz_convert("UTC").to_series().diff().dropna()
if not utc_steps.eq(pd.Timedelta(hours=1)).all():
    raise ValueError("The cleaned dataset is not a continuous hourly time series.")

print("Shape:", df.shape)
print("Period:", df.index.min(), "to", df.index.max())
print("Hourly UTC continuity: verified")

Shape: (35064, 20)
Period: 2015-01-01 00:00:00+01:00 to 2018-12-31 23:00:00+01:00
Hourly UTC continuity: verified


## 3. Construct origin-safe exogenous summaries

For each selected variable, four values are made available at the forecasting origin:

1. the most recent realised value at 11:00 on $D-1$;
2. its 24-hour lag;
3. its 168-hour lag; and
4. the backward-looking mean over the latest 24 observations, including the origin observation.

These columns are calculated chronologically and are sampled only at the forecast origin.

In [17]:
def add_origin_safe_exogenous_features(frame, variables):
    result = frame.copy()
    feature_names = []

    for variable in variables:
        current = variable
        lag24 = f"{variable}_lag24"
        lag168 = f"{variable}_lag168"
        roll24 = f"{variable}_roll24_mean"

        result[lag24] = result[variable].shift(24)
        result[lag168] = result[variable].shift(168)
        result[roll24] = result[variable].rolling(
            window=24, min_periods=24
        ).mean()

        feature_names.extend([current, lag24, lag168, roll24])

    return result, feature_names


df_features, EXOGENOUS_FEATURES = add_origin_safe_exogenous_features(
    df, EXOGENOUS_VARIABLES
)

assert len(EXOGENOUS_FEATURES) == 28
display(pd.DataFrame({"exogenous_feature": EXOGENOUS_FEATURES}))

,exogenous_feature
0,total_load_actual
1,total_load_actual_lag24
2,total_load_actual_lag168
3,total_load_actual_roll24_mean
4,gen_wind_onshore
5,gen_wind_onshore_lag24
6,gen_wind_onshore_lag168
7,gen_wind_onshore_roll24_mean
8,gen_solar
9,gen_solar_lag24


## 4. Delivery-day calendar encoding

In [19]:
def delivery_calendar(delivery_day):
    '''Return deterministic calendar descriptors for delivery day D.'''
    return np.asarray([
        np.sin(2.0 * np.pi * delivery_day.dayofweek / 7.0),
        np.cos(2.0 * np.pi * delivery_day.dayofweek / 7.0),
        np.sin(2.0 * np.pi * (delivery_day.month - 1) / 12.0),
        np.cos(2.0 * np.pi * (delivery_day.month - 1) / 12.0),
        float(delivery_day.dayofweek >= 5),
    ], dtype=np.float32)

## 5. Construct one market-consistent sample per delivery day

The availability rules differ by variable type:

- **Realised load, generation and weather:** observations are restricted to timestamps at or before 11:00 on $D-1$.
- **Day-ahead price:** the complete price curve for $D-1$ is already published before the forecast for $D$ is issued. The 168-price history may therefore end at 23:00 on $D-1$, although some of those prices have delivery timestamps after the 11:00 forecast origin.
- **Calendar variables:** characteristics of delivery day $D$ are known in advance.

Days containing 23 or 25 local delivery hours are excluded because the common forecasting task uses a fixed 24-output vector. The local forecast origin is built explicitly as 11:00; elapsed-hour arithmetic is avoided because it can shift the clock time around DST transitions.

In [21]:
def local_forecast_origin(delivery_day, hour=CUTOFF_HOUR):
    '''Construct 11:00 local time on the calendar day preceding delivery.'''
    previous_date = (delivery_day - pd.DateOffset(days=1)).date()
    return pd.Timestamp(
        year=previous_date.year,
        month=previous_date.month,
        day=previous_date.day,
        hour=hour,
        tz=TIMEZONE,
    )


def make_daily_samples(frame, exogenous_features):
    x_tabular = []
    x_sequence = []
    targets = []
    metadata_rows = []
    target_timestamp_rows = []
    excluded_rows = []

    local_days = pd.DatetimeIndex(frame.index.normalize().unique()).sort_values()

    for delivery_day in local_days:
        target_index = frame.index[frame.index.normalize() == delivery_day]

        if len(target_index) != HORIZON:
            excluded_rows.append({
                "delivery_day": delivery_day.strftime("%Y-%m-%d"),
                "reason": "non_24_hour_delivery_day",
                "observed_delivery_hours": len(target_index),
            })
            continue

        forecast_origin = local_forecast_origin(delivery_day)
        price_history_end = target_index[0] - pd.Timedelta(hours=1)
        price_history = frame.loc[:price_history_end, TARGET].tail(PRICE_WINDOW)
        origin_rows = frame.loc[:forecast_origin, exogenous_features]

        if len(price_history) != PRICE_WINDOW:
            excluded_rows.append({
                "delivery_day": delivery_day.strftime("%Y-%m-%d"),
                "reason": "insufficient_price_history",
                "observed_delivery_hours": len(target_index),
            })
            continue
        if origin_rows.empty:
            excluded_rows.append({
                "delivery_day": delivery_day.strftime("%Y-%m-%d"),
                "reason": "no_exogenous_origin_row",
                "observed_delivery_hours": len(target_index),
            })
            continue

        origin_exogenous = origin_rows.iloc[-1]
        if origin_exogenous.name != forecast_origin:
            raise ValueError(
                f"No observation exactly at forecast origin {forecast_origin}."
            )

        target = frame.loc[target_index, TARGET]
        if (
            price_history.isna().any()
            or origin_exogenous.isna().any()
            or target.isna().any()
        ):
            excluded_rows.append({
                "delivery_day": delivery_day.strftime("%Y-%m-%d"),
                "reason": "missing_required_value",
                "observed_delivery_hours": len(target_index),
            })
            continue

        calendar = delivery_calendar(delivery_day)
        static_features = np.concatenate([
            origin_exogenous.to_numpy(dtype=np.float32), calendar
        ])
        price_values = price_history.to_numpy(dtype=np.float32)

        x_tabular.append(np.concatenate([price_values, static_features]))
        x_sequence.append(np.column_stack([
            price_values,
            np.repeat(static_features[None, :], PRICE_WINDOW, axis=0),
        ]))
        targets.append(target.to_numpy(dtype=np.float32))

        sample_id = len(metadata_rows)
        metadata_rows.append({
            "sample_id": sample_id,
            "delivery_day": delivery_day.strftime("%Y-%m-%d"),
            "delivery_year": delivery_day.year,
            "forecast_origin": forecast_origin.isoformat(),
            "price_history_start": price_history.index[0].isoformat(),
            "price_history_end": price_history.index[-1].isoformat(),
        })

        for horizon, timestamp in enumerate(target_index, start=1):
            target_timestamp_rows.append({
                "sample_id": sample_id,
                "delivery_day": delivery_day.strftime("%Y-%m-%d"),
                "horizon": horizon,
                "target_timestamp": timestamp.isoformat(),
            })

    return {
        "X_tabular": np.asarray(x_tabular, dtype=np.float32),
        "X_sequence": np.asarray(x_sequence, dtype=np.float32),
        "y": np.asarray(targets, dtype=np.float32),
        "metadata": pd.DataFrame(metadata_rows),
        "target_timestamps": pd.DataFrame(target_timestamp_rows),
        "excluded_days": pd.DataFrame(excluded_rows),
    }


samples = make_daily_samples(df_features, EXOGENOUS_FEATURES)
print("Tabular shape:", samples["X_tabular"].shape)
print("Sequence shape:", samples["X_sequence"].shape)
print("Target shape:", samples["y"].shape)
display(samples["metadata"].groupby("delivery_year").size().rename("samples"))

Tabular shape: (1445, 201)
Sequence shape: (1445, 168, 34)
Target shape: (1445, 24)


delivery_year
2015    355
2016    364
2017    363
2018    363
Name: samples, dtype: int64

## 6. Feature names and dimensional validation

The tabular representation contains 201 predictors:

**168 historical prices + 28 exogenous summaries + 5 calendar variables = 201 predictors.**

The recurrent representation contains the same information as a **168 × 34** tensor. Each position contains one historical-price value plus the 33 origin-level exogenous and calendar values repeated across the sequence. This representation does not imply that 34 separate variables were observed throughout the entire 168-hour period.

In [23]:
PRICE_FEATURES = [
    f"price_history_{position:03d}"
    for position in range(1, PRICE_WINDOW + 1)
]
TABULAR_FEATURES = PRICE_FEATURES + EXOGENOUS_FEATURES + CALENDAR_FEATURES
SEQUENCE_FEATURES = [TARGET] + EXOGENOUS_FEATURES + CALENDAR_FEATURES

expected_tabular_shape = (len(samples["metadata"]), 201)
expected_sequence_shape = (len(samples["metadata"]), 168, 34)
expected_target_shape = (len(samples["metadata"]), 24)

if samples["X_tabular"].shape != expected_tabular_shape:
    raise ValueError(
        f"Unexpected tabular shape: {samples['X_tabular'].shape}; "
        f"expected {expected_tabular_shape}."
    )
if samples["X_sequence"].shape != expected_sequence_shape:
    raise ValueError(
        f"Unexpected sequence shape: {samples['X_sequence'].shape}; "
        f"expected {expected_sequence_shape}."
    )
if samples["y"].shape != expected_target_shape:
    raise ValueError(
        f"Unexpected target shape: {samples['y'].shape}; "
        f"expected {expected_target_shape}."
    )
if len(samples["target_timestamps"]) != len(samples["metadata"]) * HORIZON:
    raise ValueError("Target-timestamp metadata are incomplete.")

feature_audit = pd.DataFrame({
    "component": [
        "Historical prices",
        "Origin-safe exogenous summaries",
        "Delivery-day calendar",
        "Tabular predictors",
        "Sequence channels",
        "Output horizons",
    ],
    "count": [168, 28, 5, 201, 34, 24],
})
display(feature_audit)

,component,count
0,Historical prices,168
1,Origin-safe exogenous summaries,28
2,Delivery-day calendar,5
3,Tabular predictors,201
4,Sequence channels,34
5,Output horizons,24


## 7. Chronological partition audit

The saved feature arrays remain a single chronologically ordered dataset. Subsequent notebooks select training and evaluation samples by `delivery_year`:

- train on 2015–2016 and predict 2017 for model ranking, diversity analysis and stacking estimation;
- train on 2015–2017 and predict 2018 for the final untouched evaluation.

No scaler or other fitted transformation is applied here; such transformations must be fitted using only the corresponding training years.

In [25]:
partition_summary = (
    samples["metadata"]
    .groupby("delivery_year", as_index=False)
    .agg(
        samples=("sample_id", "size"),
        first_delivery_day=("delivery_day", "min"),
        last_delivery_day=("delivery_day", "max"),
    )
)

if set(partition_summary["delivery_year"]) != {2015, 2016, 2017, 2018}:
    raise ValueError("Expected feature samples for delivery years 2015–2018.")

display(partition_summary)

,delivery_year,samples,first_delivery_day,last_delivery_day
0,2015,355,2015-01-09,2015-12-31
1,2016,364,2016-01-01,2016-12-31
2,2017,363,2017-01-01,2017-12-31
3,2018,363,2018-01-01,2018-12-31


## 8. Save the feature dataset and audit files

In [27]:
np.savez_compressed(
    FEATURE_FILE,
    X_tabular=samples["X_tabular"],
    X_sequence=samples["X_sequence"],
    y=samples["y"],
)

samples["metadata"].to_csv(
    FEATURE_DIR / "sample_metadata.csv", index=False
)
samples["target_timestamps"].to_csv(
    FEATURE_DIR / "target_timestamps.csv", index=False
)
pd.DataFrame({"feature": TABULAR_FEATURES}).to_csv(
    FEATURE_DIR / "tabular_feature_names.csv", index=False
)
pd.DataFrame({"feature": SEQUENCE_FEATURES}).to_csv(
    FEATURE_DIR / "sequence_feature_names.csv", index=False
)
samples["excluded_days"].to_csv(
    AUDIT_DIR / "excluded_delivery_days.csv", index=False
)

feature_summary = pd.DataFrame([
    {"item": "eligible_daily_samples", "value": len(samples["metadata"])},
    {"item": "tabular_predictors", "value": samples["X_tabular"].shape[1]},
    {"item": "sequence_length", "value": samples["X_sequence"].shape[1]},
    {"item": "sequence_channels", "value": samples["X_sequence"].shape[2]},
    {"item": "target_horizons", "value": samples["y"].shape[1]},
    {"item": "excluded_delivery_days", "value": len(samples["excluded_days"])},
])
feature_summary.to_csv(
    AUDIT_DIR / "feature_construction_summary.csv", index=False
)

manifest = {
    "timezone": TIMEZONE,
    "forecast_origin_local_hour": CUTOFF_HOUR,
    "price_window": PRICE_WINDOW,
    "forecast_horizon": HORIZON,
    "target": TARGET,
    "exogenous_variables": EXOGENOUS_VARIABLES,
    "calendar_features": CALENDAR_FEATURES,
    "tabular_shape": list(samples["X_tabular"].shape),
    "sequence_shape": list(samples["X_sequence"].shape),
    "target_shape": list(samples["y"].shape),
    "dtype": "float32",
}
with (FEATURE_DIR / "feature_manifest.json").open("w", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2)

display(feature_summary)
print(f"Feature arrays saved to: {FEATURE_FILE}")
print(f"Metadata saved to: {FEATURE_DIR}")
print(f"Audits saved to: {AUDIT_DIR}")

,item,value
0,eligible_daily_samples,1445
1,tabular_predictors,201
2,sequence_length,168
3,sequence_channels,34
4,target_horizons,24
5,excluded_delivery_days,16


Feature arrays saved to: /Users/fatimarodrigues/Documents/Aulas/MINDD/Trabs-Pratico/2025-26/TP2-2025/git-price-forecasting/electricity-price-forecasting-revisão/initial-weather/data/features/day_ahead_features.npz
Metadata saved to: /Users/fatimarodrigues/Documents/Aulas/MINDD/Trabs-Pratico/2025-26/TP2-2025/git-price-forecasting/electricity-price-forecasting-revisão/initial-weather/data/features
Audits saved to: /Users/fatimarodrigues/Documents/Aulas/MINDD/Trabs-Pratico/2025-26/TP2-2025/git-price-forecasting/electricity-price-forecasting-revisão/initial-weather/outputs/audits


## 9. Reloading example for subsequent notebooks

The following pattern can be used by the model-training notebook. The year masks are derived solely from delivery-day metadata.

In [29]:
loaded = np.load(FEATURE_FILE)
metadata = pd.read_csv(FEATURE_DIR / "sample_metadata.csv")

X_tabular = loaded["X_tabular"]
X_sequence = loaded["X_sequence"]
y = loaded["y"]

mask_2017 = metadata["delivery_year"].eq(2017).to_numpy()
mask_2018 = metadata["delivery_year"].eq(2018).to_numpy()

print("2017 tabular/target:", X_tabular[mask_2017].shape, y[mask_2017].shape)
print("2018 tabular/target:", X_tabular[mask_2018].shape, y[mask_2018].shape)

2017 tabular/target: (363, 201) (363, 24)
2018 tabular/target: (363, 201) (363, 24)


## Reproducibility note

The feature arrays are intentionally unscaled. For each forecasting experiment, scaling parameters must be estimated from the applicable training years only and then applied unchanged to the corresponding prediction year. The 2018 samples must not be used for feature selection, model ranking, hyperparameter selection, ensemble composition or stacking estimation.